# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
!pip install langchain langchain-community langchain-experimental langchain-openai langchain-classic

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import Optional

# Legacy chains/prompts → langchain-classic
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate

# BabyAGI still in langchain-experimental
from langchain_experimental.autonomous_agents import BabyAGI

# OpenAI integrations
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [3]:
# # %pip install faiss-cpu > /dev/null
# # %pip install google-search-results > /dev/null
# from langchain.docstore import InMemoryDocstore
# from langchain_community.vectorstores import FAISS

In [4]:
!pip install faiss-cpu langchain-community

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore  # Updated path
from langchain_community.vectorstores import FAISS

In [6]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [7]:
# Define your embedding model
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings()

# Initialize the vectorstore as empty
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(
    embedding_function=embeddings_model,  # Full embeddings object, not .embed_query
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [8]:
!pip install duckduckgo-search

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.9 MB 217.7 kB/s eta 0:00:16
   ----- -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_classic.agents import AgentExecutor, Tool, ZeroShotAgent  # Legacy agents
from langchain_classic.chains import LLMChain  # Legacy chains
from langchain_classic.prompts import PromptTemplate  # Legacy prompts
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), prompt=todo_prompt)
search = DuckDuckGoSearchRun()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [11]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

C:\Users\sisaz\AppData\Local\Temp\ipykernel_13088\1591959347.py:4: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [12]:
OBJECTIVE = "Write a weather report for SF today"

In [13]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [14]:
baby_agi({"objective": OBJECTIVE})

C:\Users\sisaz\AppData\Local\Temp\ipykernel_13088\3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Question: What is the weather report for San Francisco today?
Thought: I need to find the current weather conditions for San Francisco to provide an accurate report.
Action: Search
Action Input: 'San Francisco weather today'
Observation: 2 weeks ago - San Francisco, CA Weather Forecast, with current conditions, wind, air quality, and what to expect for the next 3 days. April 12, 2026 - San Francisco CA 37.77°N 122.41°W (Elev. 1 week ago - How much will it rain or snow today in San Francisco? None expected · What is the current wind speed and direction in San Francisco? 22 km/h W · What is the current humidity level in San Francisco? 79% Are there any active weather alerts in San Francisco? No · What is the UV index right now in San Francisco? 1 month ago - Zone Area Forecast for San Francisco County · Forecast Discussion · Printable Forecast · Text Only Forecast ·

{'objective': 'Write a weather report for SF today'}